# Calibración del baseline contra el BUR3 — Indonesia

Ajusta el baseline de SISEPUEDE (`primary_id 0`) **sector por sector** para que
reproduzca el inventario nacional del **BUR3** (UNFCCC 2022), y exporta una **base
calibrada final** (`IDN_ssp_input_calibrated_<fecha>.csv`).

**Mecanismo.** Por cada subsector se calcula un factor multiplicativo en el año ancla
(2019, el inventario más reciente y con el desglose completo) y se aplica a los campos
de *entrada* de ese subsector. Como los factores de emisión entran de forma lineal,
escalar por `f` escala las emisiones del subsector ~`f`, así que el resultado calibrado
se **predice analíticamente** (rápido) y, opcionalmente, se confirma con una re-corrida.

| Palanca | Campos que escala | Subsectores |
|---|---|---|
| `ef` | `ef_<sub>_*` | fgtv, ippu, trns, soil, waso, lsmm, trww, lvst, agrc, frst, lndu |
| `demand` | `scalar_<sub>_*energy_demand*` | inen, scoe |
| `manual` | — (se reporta pero **no** se aplica) | entc, ccsq |

**FOLU / turba (peat).** El FOLU de Indonesia está dominado por turba (~854 Mt en 2019)
y el modelo no tiene variable explícita de peat. Se mantiene el sumidero forestal
(`frst`) y se escala `lndu` para que `frst + lndu·f` llegue al FOLU del BUR3; la turba
queda "absorbida" en `lndu` → se marca **STRUCTURAL_GAP** (no es una calibración física).

**`entc` (electricidad).** No tiene campos `ef_` ni un escalar de demanda simple (la
demanda eléctrica se deriva de todos los sectores). Se deja como palanca **manual**:
para cerrar su brecha hay que tocar la mezcla de generación (`frac_entc_*`), no un EF.


In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys
from pathlib import Path
from datetime import date
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml

pd.set_option("display.width", 240)
pd.set_option("display.max_columns", 40)

# make shared_scripts importable and load the calibration helper
sys.path.insert(0, str(Path.cwd() / "shared_scripts"))
import bur3_calibration as cb

In [ ]:
# Paths (mirror indonesia_manager_wb.ipynb)
NOTEBOOKS_DIR_PATH = Path.cwd()
SSP_MODELING_DIR_PATH = NOTEBOOKS_DIR_PATH.parent
DATA_DIR_PATH = SSP_MODELING_DIR_PATH / "input_data"
CONFIG_DIR_PATH = NOTEBOOKS_DIR_PATH / "config_files"

# Config
config = yaml.safe_load(open(CONFIG_DIR_PATH / "config.yaml"))
country_name = config["country_name"]
ssp_input_file_name = config["ssp_input_file_name"]
sim_end_year = config.get("sim_end_year", 2050)
energy_model_flag = config["energy_model_flag"]
print(f"country={country_name}  input={ssp_input_file_name}  end_year={sim_end_year}  energy_model={energy_model_flag}")

# BUR3 reference + anchor year
CALIB_YEAR = 2019
ref = cb.load_reference(CONFIG_DIR_PATH / "indonesia_bur3_inventory.yaml")
print("BUR3 reference loaded; anchor year =", CALIB_YEAR)

In [ ]:
# Resolve the SISEPUEDE output directory (where the last baseline run was written).
# Lightweight: no Julia, no model run -- we only read MODEL_OUTPUT.csv from disk.
try:
    import sisepuede.manager.sisepuede_file_structure as sfs
    _fs = sfs.SISEPUEDEFileStructure(initialize_directories=False)
    DIR_OUT = str(_fs.dir_out)
except Exception as e:
    import sisepuede
    DIR_OUT = str(Path(sisepuede.__file__).resolve().parent / "out")
    print("fallback dir_out:", e)
print("SISEPUEDE out dir:", DIR_OUT)

## Paso 1 — Baseline actual vs BUR3

Carga el baseline (`primary_id 0`) de la **última corrida** y lo agrega a sectores IPCC.
El total con FOLU cuadra casi con el BUR3, pero por errores que se cancelan (Energy/IPPU
altos, FOLU bajo): mira el total **sin** FOLU para ver el desajuste real.

In [ ]:
baseline_output = cb.find_latest_output(DIR_OUT)
print("baseline desde:", Path(baseline_output).parent.parent.name)

base = cb.load_baseline(baseline_output)          # year + subsector emissions, Mt CO2e
agg = cb.aggregate_ipcc(base)

key_years = [2015, 2019, 2030, 2050]
show = agg[agg.year.isin(key_years)].round(1).set_index("year")
print("\n=== Baseline por sector IPCC (Mt CO2e) ===")
print(show[["Energy","IPPU","Agriculture","FOLU","Waste","Total_incl_FOLU","Total_excl_FOLU"]].to_string())
print("\nBUR3 2019:", {k: ref['sector_targets_2019'][k] for k in ['Energy','IPPU','Agriculture','FOLU','Waste','Total_incl_FOLU','Total_excl_FOLU']})

In [ ]:
# Barras: modelo vs BUR3 por sector (año ancla)
sectors = ["Energy","IPPU","Agriculture","FOLU","Waste"]
m = agg[agg.year==CALIB_YEAR].iloc[0]
model_v = [m[s] for s in sectors]
bur3_v  = [ref["sector_targets_2019"][s] for s in sectors]
x = np.arange(len(sectors)); w = 0.38
fig, ax = plt.subplots(figsize=(10,5))
ax.bar(x-w/2, model_v, w, label="Modelo", color="#4C78A8")
ax.bar(x+w/2, bur3_v,  w, label="BUR3",   color="#E45756")
ax.set_xticks(x); ax.set_xticklabels(sectors); ax.set_ylabel("Mt CO2e")
ax.set_title(f"Baseline vs BUR3 por sector — {CALIB_YEAR}"); ax.legend()
for i,(a,b) in enumerate(zip(model_v,bur3_v)):
    ax.text(i-w/2, a, f"{a:.0f}", ha="center", va="bottom", fontsize=8)
    ax.text(i+w/2, b, f"{b:.0f}", ha="center", va="bottom", fontsize=8)
plt.tight_layout(); plt.show()

## Paso 2 — Diagnóstico por subsector y factores

`factor = BUR3 / modelo` en el año ancla. `applied=True` = se escribe en los inputs;
`manual` = se reporta pero no se aplica. Ordena por tamaño de la brecha para ver qué
domina el error (spoiler: `fgtv` ~15×, `ippu` ~3×).

In [ ]:
comp = cb.build_comparison(base, ref, CALIB_YEAR)
disp = comp.copy()
disp["gap_Mt"] = (disp[f"model_{CALIB_YEAR}"] - disp[f"bur3_{CALIB_YEAR}"]).round(1)
disp["factor"] = disp["factor"].round(4)
cols = ["ipcc_sector","subsector","lever",f"model_{CALIB_YEAR}",f"bur3_{CALIB_YEAR}","gap_Mt","factor","applied","note"]
disp[cols]

In [ ]:
# Brecha por subsector (modelo - BUR3), ordenada
d = comp.copy()
d["gap"] = d[f"model_{CALIB_YEAR}"] - d[f"bur3_{CALIB_YEAR}"]
d = d.sort_values("gap")
colors = ["#E45756" if g>0 else "#54A24B" for g in d["gap"]]
fig, ax = plt.subplots(figsize=(10,6))
ax.barh(d["subsector"], d["gap"], color=colors)
ax.axvline(0, color="k", lw=0.8)
ax.set_xlabel("Modelo − BUR3 (Mt CO2e)   [rojo = modelo sobrestima]")
ax.set_title(f"Brecha por subsector — {CALIB_YEAR}")
plt.tight_layout(); plt.show()

## Paso 3 — Perillas de calibración (editable)

Los factores se autocalculan. Para **sobrescribir** uno a mano antes de aplicar, edita
`comp` (ej. forzar `entc` o suavizar `lndu`):

```python
comp.loc[comp.subsector == "entc", ["factor", "applied"]] = [0.88, True]  # cuidado: entc no tiene ef_, no tendría efecto
comp.loc[comp.subsector == "lndu", "factor"] = 1.5   # calibración FOLU más conservadora
```


In [ ]:
calib_factors = cb.factors_dict(comp)
print("Factores por subsector (ancla %d):" % CALIB_YEAR)
for s, f in calib_factors.items():
    lev = comp.loc[comp.subsector==s, "lever"].iloc[0]
    app = comp.loc[comp.subsector==s, "applied"].iloc[0]
    tag = "" if app else "  (manual, NO se aplica)"
    print(f"  {s:5s} [{lev:6s}] x{f:7.4f}{tag}")

## Paso 4 — FOLU / turba (peat)

El inventario BUR3 de FOLU 2019 está dominado por turba. El modelo no lo representa
mecánicamente, así que `lndu` se infla para alcanzar el neto (marca `STRUCTURAL_GAP`).
Si prefieres calibrar FOLU a un año **sin** incendios o a la meta AFOLU del NDC, cambia
`folu_2019.net_total` en el YAML o fija el factor de `lndu` a mano en el Paso 3.

## Paso 4b — `entc` / generación eléctrica (límite estructural, **no** calibrado)

`entc` es la única brecha que queda abierta a propósito. Tras la calibración (iter2) la
re-corrida real da **entc 2019 ≈ 206 Mt vs BUR3 289** (−29%). Se diagnosticó a fondo y se
decidió **dejarlo como está**, porque no se cierra con una sola perilla física.

**Descomposición de `entc` 2019 (206 Mt):**

| componente | Mt CO2e | nota |
|---|---:|---|
| Generación eléctrica (centrales) | 139 | carbón 133 |
| Minería/extracción de combustibles | 35 | carbón/gas/petróleo |
| Procesamiento de biomasa | 31 | refinación bioenergía |

**Por qué no se cierra con calibración:**

1. **La mezcla de generación del modelo está demasiado limpia.** En 2019 el modelo tiene
   ~33 % cero-carbono (hidro 17 %, **nuclear 4.5 % — inexistente en Indonesia**, eólica+solar
   ~9 %) cuando la realidad fue ~14 %. Ese exceso desplaza al fósil y hunde las emisiones.
   Corregir la mezcla a shares reales (`nemomod_entc_frac_min_share_production_pp_*`) solo
   recupera ~+25 Mt (→ ~234) y **reescribe toda la trayectoria energética 2015–2050**, no
   solo el año ancla: es una decisión de modelado, no un ajuste de calibración.
2. **Acoplamiento de demanda.** Los recortes de `inen`/`scoe` (palanca `demand`) bajaron la
   demanda eléctrica, y con ella la generación y sus emisiones. Parte del déficit de `entc`
   es un efecto colateral de calibrar la combustión directa de esos sectores.
3. **Alcance de BUR3 1A1.** "Energy Industries" incluye refinerías y uso propio del sector
   energético; parte de eso el modelo lo contabiliza en otras categorías.

**Consecuencia:** Energy queda ~−13 % (556 vs 636) y el total 2019 ~−4.5 % (1762 vs 1845).
El resto de sectores (IPPU, Agricultura, FOLU, Residuos) cuadra. Si en el futuro quieres
cerrar `entc`, la vía física es ajustar la mezcla de generación (perilla de arriba) y/o
la demanda eléctrica, re-corriendo e iterando como se hizo con FOLU — no un factor `ef`.


In [ ]:
folu = ref["folu_2019"]
print("FOLU 2019 (BUR3), Mt CO2e:")
for k,v in folu.items(): print(f"  {k:28s} {v}")
frst_m = base.loc[base.year==CALIB_YEAR, "frst"].iloc[0]
lndu_m = base.loc[base.year==CALIB_YEAR, "lndu"].iloc[0]
print(f"\nModelo {CALIB_YEAR}:  frst={frst_m:.1f} (sumidero, se mantiene)   lndu={lndu_m:.1f}")
print(f"Factor lndu = (FOLU_target - frst) / lndu = ({folu['net_total']} - ({frst_m:.1f})) / {lndu_m:.1f} "
      f"= {comp.loc[comp.subsector=='lndu','factor'].iloc[0]:.3f}")

## Paso 5 — Aplicar a los inputs y predecir el resultado

Escala los campos de entrada y predice las emisiones calibradas (modelo × factor).
Todo cuadra con el BUR3 salvo `entc` (manual): su brecha es el residual del total.

In [ ]:
inp = pd.read_csv(DATA_DIR_PATH / ssp_input_file_name)
cal, change_log = cb.apply_factors(inp, comp)
print("Campos escalados por subsector:")
print(change_log.to_string(index=False))

pred = cb.predict(base, comp)
agg_before = cb.aggregate_ipcc(base)
agg_after  = cb.aggregate_ipcc(pred)
b = agg_before[agg_before.year==CALIB_YEAR].iloc[0]
a = agg_after[agg_after.year==CALIB_YEAR].iloc[0]
print(f"\n=== Sectores {CALIB_YEAR}: antes -> después (BUR3) ===")
for s in ["Energy","IPPU","Agriculture","FOLU","Waste","Total_incl_FOLU"]:
    t = ref["sector_targets_2019"].get(s,"")
    print(f"  {s:16s} {b[s]:8.1f} -> {a[s]:8.1f}   (BUR3 {t})")
print("\n(residual del total = brecha de entc, que es palanca manual)")

In [ ]:
# Trayectoria del total: baseline original vs calibrado (predicho)
fig, ax = plt.subplots(figsize=(11,5))
ax.plot(agg_before.year, agg_before.Total_incl_FOLU, "--", color="#4C78A8", label="Baseline original")
ax.plot(agg_after.year,  agg_after.Total_incl_FOLU,  "-",  color="#54A24B", label="Calibrado (predicho)")
ax.scatter([2019],[ref["sector_targets_2019"]["Total_incl_FOLU"]], color="#E45756", zorder=5, label="BUR3 2019")
ax.scatter([2030],[ref["ndc_bau"]["y2030"]["Total"]], marker="s", color="#B279A2", zorder=5, label="NDC BAU 2030")
ax.set_xlabel("Año"); ax.set_ylabel("Mt CO2e (incl. FOLU)")
ax.set_title("Emisiones totales: baseline vs calibrado"); ax.legend()
plt.tight_layout(); plt.show()

## Paso 6 — Exportar la base calibrada final

Escribe `IDN_ssp_input_calibrated_<hoy>.csv`. Para usarla en escenarios, apunta
`config.yaml → ssp_input_file_name` al archivo nuevo y re-corre `indonesia_manager_wb.ipynb`.

In [ ]:
out_name = f"IDN_ssp_input_calibrated_{date.today():%Y%m%d}.csv"
out_path = DATA_DIR_PATH / out_name
cal.to_csv(out_path, index=False)
print("Base calibrada escrita en:")
print(" ", out_path)
print(f"\nSiguiente paso: en config.yaml  ssp_input_file_name: \"{out_name}\"  y re-correr el manager.")

## Paso 7 (opcional) — Verificación con re-corrida completa

La predicción del Paso 5 es analítica (lineal). Para **confirmar** —sobre todo las
palancas `demand`, que pasan por el modelo de energía (Julia)— apunta `config.yaml` al
archivo nuevo, re-corre el baseline en `indonesia_manager_wb.ipynb`, y vuelve a correr
la celda de abajo: busca la corrida más nueva y compara **real vs predicho**.

In [ ]:
RUN_VERIFICATION = False  # pon True después de re-correr el manager con la base calibrada

if RUN_VERIFICATION:
    new_out = cb.find_latest_output(DIR_OUT)
    assert Path(new_out) != Path(baseline_output), "No hay una corrida más nueva; re-corre el manager primero."
    base_new = cb.load_baseline(new_out)
    agg_real = cb.aggregate_ipcc(base_new)
    r = agg_real[agg_real.year==CALIB_YEAR].iloc[0]
    print(f"=== Verificación {CALIB_YEAR}: predicho vs real vs BUR3 ===")
    for s in ["Energy","IPPU","Agriculture","FOLU","Waste","Total_incl_FOLU"]:
        t = ref["sector_targets_2019"].get(s,"")
        print(f"  {s:16s} pred {a[s]:8.1f} | real {r[s]:8.1f} | BUR3 {t}")
else:
    print("RUN_VERIFICATION=False — pon True tras re-correr el manager con la base calibrada.")